<a href="https://colab.research.google.com/github/blackwatermelon0000/NLP/blob/main/LAB_02_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

In [5]:
# Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

import pdfplumber
import re
from collections import Counter

from google.colab import files
uploaded = files.upload()

pdf_path = "Regex, Tokenization & Sentence Segmentation.pdf"


pages_text = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text is None:   # some pages might be blank/images, just handle that
            text = ""
        pages_text.append(text)

print("Total pages extracted:", len(pages_text))


# (b) Merge all page strings into a single string. Use re.sub() to remove all punctuation and digits, convert to lowercase, and split on whitespace to produce a token list. Print the total token count.

full_text = " ".join(pages_text)

# removing anything that's not a letter or a space
cleaned_text = re.sub(r'[^a-zA-Z\s]', '', full_text)
cleaned_text = cleaned_text.lower()

tokens = cleaned_text.split()

print("Total tokens:", len(tokens))


# (c) Build a word frequency dictionary from the token list. Remove at least 20 common stopwords. Print the top 10 most frequent words along with their counts in descending order.

stopwords = {
    "the", "is", "in", "and", "to", "of", "a", "an", "it", "for",
    "on", "with", "as", "was", "were", "are", "this", "that", "be",
    "by", "at", "or", "from", "but", "not", "have", "has", "had"
}

filtered_tokens = [word for word in tokens if word not in stopwords]

freq = Counter(filtered_tokens)  # counts everything for us, easier than a manual dict loop

top_10 = freq.most_common(10)

print("\nTop 10 most frequent words:")
for word, count in top_10:
    print(word, "-", count)


# (d) Accept a keyword from the user via input(). Search through the page-wise list, split each page's text into sentences, and print every sentence containing the keyword along with its page number. If no match is found, print an appropriate message.

keyword = input("\nEnter a keyword to search: ").lower()

found = False

for page_num, page_text in enumerate(pages_text, start=1):
    # basic sentence split on . ? ! -- not perfect but works fine for this
    sentences = re.split(r'(?<=[.!?])\s+', page_text)
    for sentence in sentences:
        if keyword in sentence.lower():
            print(f"Page {page_num}: {sentence.strip()}")
            found = True

if not found:
    print("No sentences found with that keyword.")

Saving Regex, Tokenization & Sentence Segmentation.pdf to Regex, Tokenization & Sentence Segmentation.pdf
Total pages extracted: 4
Total tokens: 1243

Top 10 most frequent words:
used - 24
nlp - 24
pipeline - 20
text - 16
matters - 15
where - 15
why - 14
like - 14
can - 13
token - 13

Enter a keyword to search: Regex
Page 1: Part A: Regex Fundamentals (Foundation)
A1.
Page 1: What is Regex?
Page 1: Regex (Regular Expression) is a special pattern language.
Page 1: Without regex, we would have to search everything by hand, which is slow and full of mistakes.
Page 1: Where it is used in NLP pipeline: Regex is used in the preprocessing step.
Page 1: Before any NLP model works on
the data, we clean the text or pull out specific information (like emails or phone numbers) using regex.
Page 1: Character Classes
Character classes tell regex what type of character to look for.
Page 1: Symbol Meaning
\d Any digit (0-9)
\w Any word character (letters, digits, underscore)
\s Space, tab, or new line

# Task 2: Structured Page Metadata Extraction and CSV Export

In [ ]:
# (a) Open a multi-page PDF (minimum 5 pages). For each page, build a dictionary with exactly these keys: page_number, raw_text, word_count, and char_count (excluding spaces). Collect all dictionaries into a list called pages_data and print the first 2 entries.

import pdfplumber
import csv

pdf_path = "Regex, Tokenization & Sentence Segmentation.pdf"
pages_data = []

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()
        if text is None:
            text = ""  # just in case a page has no extractable text

        word_count = len(text.split())
        char_count = len(text.replace(" ", ""))  # excluding spaces like asked

        page_dict = {
            "page_number": i,
            "raw_text": text,
            "word_count": word_count,
            "char_count": char_count
        }

        pages_data.append(page_dict)

print("First 2 entries:")
for entry in pages_data[:2]:
    print(entry)


# (b) From pages_data, identify and print: (i) the page with the highest word count and its full content, (ii) the page with the lowest non-zero word count, and (iii) the average word count across all pages rounded to 2 decimal places. Handle pages that return zero words.

# (i) highest word count page
highest_page = max(pages_data, key=lambda x: x["word_count"])
print("\nPage with highest word count:", highest_page["page_number"])
print("Content:\n", highest_page["raw_text"])

# (ii) lowest NON-ZERO word count -- so we filter out the zero ones first
non_zero_pages = [p for p in pages_data if p["word_count"] > 0]

if non_zero_pages:
    lowest_page = min(non_zero_pages, key=lambda x: x["word_count"])
    print("\nPage with lowest non-zero word count:", lowest_page["page_number"], "-", lowest_page["word_count"], "words")
else:
    print("\nAll pages have zero words, nothing to compare here.")

# (iii) average word count, rounded to 2 decimals
total_words = sum(p["word_count"] for p in pages_data)
avg_words = round(total_words / len(pages_data), 2)
print("\nAverage word count across all pages:", avg_words)


# (c) Accept a search phrase via input(). Perform a case-insensitive search across all pages. For every page containing the phrase, print its page number and how many times the phrase appears. If no page contains the phrase, raise a custom exception named PhraseNotFoundError and catch it gracefully.

class PhraseNotFoundError(Exception):
    # custom exception, just a simple pass-through
    pass

search_phrase = input("\nEnter a phrase to search for: ").lower()

matches_found = False

try:
    for page in pages_data:
        page_text_lower = page["raw_text"].lower()
        occurrences = page_text_lower.count(search_phrase)

        if occurrences > 0:
            print(f"Page {page['page_number']}: phrase found {occurrences} time(s)")
            matches_found = True

    if not matches_found:
        raise PhraseNotFoundError(f"'{search_phrase}' was not found in any page.")

except PhraseNotFoundError as e:
    print("Error:", e)


# (d) Using Python's built-in csv module (not pandas), write a file named pdf_summary.csv with columns: page_number, word_count, char_count. Then re-read that file and print its contents row by row to confirm the write was successful.

csv_filename = "pdf_summary.csv"

with open(csv_filename, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])  # header row

    for page in pages_data:
        writer.writerow([page["page_number"], page["word_count"], page["char_count"]])

print(f"\n{csv_filename} written successfully. Reading it back now:\n")

# re-reading to confirm it actually saved properly
with open(csv_filename, mode="r") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

First 2 entries:
{'page_number': 1, 'raw_text': 'Part A: Regex Fundamentals (Foundation)\nA1. What is Regex?\nRegex (Regular Expression) is a special pattern language. It is used to find specific pieces of text, such as emails,\nphone numbers, or dates, inside a big block of text.\nWhy it matters: Raw text (emails, chats, documents) contains useful information mixed with a lot of other text.\nWithout regex, we would have to search everything by hand, which is slow and full of mistakes.\nWhere it is used in NLP pipeline: Regex is used in the preprocessing step. Before any NLP model works on\nthe data, we clean the text or pull out specific information (like emails or phone numbers) using regex.\nA2. Character Classes\nCharacter classes tell regex what type of character to look for.\nSymbol Meaning\n\\d Any digit (0-9)\n\\w Any word character (letters, digits, underscore)\n\\s Space, tab, or new line\n[a-z] Any lowercase letter from a to z\n[A-Z0-9] A custom set - capital letters or digi

# Task 3: PDF Text Preprocessing Pipeline for NLP

In [ ]:
# (a) Open any PDF file and extract all text into one string. Apply the following cleaning steps in order using re.sub(): (i) remove all URLs, (ii) remove all digits, (iii) remove all non-alphabetic characters except spaces and full stops, (iv) collapse multiple consecutive spaces into a single space. Print the character count before and after cleaning.

import pdfplumber
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')  # needed once, colab usually doesn't have it by default

pdf_path = "Regex, Tokenization & Sentence Segmentation.pdf"
full_text = ""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += text + " "

char_count_before = len(full_text)

# (i) remove urls
cleaned = re.sub(r'http\S+|www\.\S+', '', full_text)

# (ii) remove digits
cleaned = re.sub(r'\d+', '', cleaned)

# (iii) remove anything that's not a letter, space or full stop
cleaned = re.sub(r'[^a-zA-Z.\s]', '', cleaned)

# (iv) collapse multiple spaces into one
cleaned = re.sub(r'\s+', ' ', cleaned).strip()

char_count_after = len(cleaned)

print("Character count before cleaning:", char_count_before)
print("Character count after cleaning:", char_count_after)


# (b) Convert the cleaned text to lowercase. Split into a list of sentences using re.split(r'[.!?]', text). Strip each sentence of leading and trailing whitespace. Remove empty strings. Print the sentence count before and after removing empty strings.

cleaned_lower = cleaned.lower()

sentences_raw = re.split(r'[.!?]', cleaned_lower)

sentence_count_before = len(sentences_raw)

# strip whitespace and drop the empty ones
sentences = [s.strip() for s in sentences_raw if s.strip() != ""]

sentence_count_after = len(sentences)

print("\nSentence count before removing empty strings:", sentence_count_before)
print("Sentence count after removing empty strings:", sentence_count_after)


# (c) Tokenize each sentence into words. Remove English stopwords using nltk.corpus.stopwords.words('english'). Discard any sentence that has fewer than 5 tokens remaining after stopword removal. Print the sentence count before and after this filter and display 3 sample sentences.

stop_words = set(stopwords.words('english'))

sentence_count_before_filter = len(sentences)

filtered_sentences = []

for sentence in sentences:
    words = sentence.split()
    words_no_stop = [w for w in words if w not in stop_words]

    if len(words_no_stop) >= 5:   # keep only if 5 or more tokens remain
        # rebuilding the sentence from the filtered words
        filtered_sentences.append(" ".join(words_no_stop))

sentence_count_after_filter = len(filtered_sentences)

print("\nSentence count before stopword filter:", sentence_count_before_filter)
print("Sentence count after stopword filter:", sentence_count_after_filter)

print("\n3 sample sentences:")
for s in filtered_sentences[:3]:
    print("-", s)


# (d) Write all remaining sentences to a file named nlp_ready_output.txt, one sentence per line. Re-read the file and print: (i) total lines written, (ii) the longest sentence by character length, and (iii) the shortest sentence by character length.

output_file = "nlp_ready_output.txt"

with open(output_file, "w") as f:
    for sentence in filtered_sentences:
        f.write(sentence + "\n")

# now re-reading it back to confirm everything saved fine
with open(output_file, "r") as f:
    lines = [line.strip() for line in f.readlines()]

print("\nTotal lines written:", len(lines))

if lines:  # just a safety check in case filtering removed everything
    longest = max(lines, key=len)
    shortest = min(lines, key=len)
    print("Longest sentence:", longest)
    print("Shortest sentence:", shortest)
else:
    print("No sentences left to compare, file is empty.")

# Task 4: DOCX Paragraph Classification and Statistical Analysis

In [ ]:
# (a) Open any .docx file using python-docx. For each paragraph, store its text and style.name in a list of dictionaries. Print the total paragraph count and all unique style names found in the document.

from docx import Document
from collections import OrderedDict

docx_path = "Lab 02 Data Acquisition for NLP.docx"
doc = Document(docx_path)

paragraphs_data = []

for para in doc.paragraphs:
    para_dict = {
        "text": para.text,
        "style": para.style.name
    }
    paragraphs_data.append(para_dict)

print("Total paragraph count:", len(paragraphs_data))

# using a set to get unique style names, then converting to list just to print nicely
unique_styles = list(set(p["style"] for p in paragraphs_data))
print("Unique style names found:", unique_styles)


# (b) Classify all paragraphs into exactly three groups: (i) Headings — style name starts with "Heading", (ii) Body — style is "Normal" or contains "Body", (iii) Other/Empty — everything else including blank paragraphs. Print the count of each group.

headings = []
body = []
other_empty = []

for p in paragraphs_data:
    style = p["style"]
    text = p["text"].strip()

    if not text:
        other_empty.append(p)  # blank paragraphs go here no matter the style
    elif style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        body.append(p)
    else:
        other_empty.append(p)

print("\nHeadings count:", len(headings))
print("Body count:", len(body))
print("Other/Empty count:", len(other_empty))


# (c) For every Body paragraph with more than 5 words, compute its word count and character count excluding spaces. Print: (i) the paragraph with the highest word count, (ii) the paragraph with the lowest word count above 5 words, and (iii) the mean word count across all qualifying body paragraphs rounded to 2 decimal places.

qualifying_body = []

for p in body:
    word_count = len(p["text"].split())
    if word_count > 5:
        char_count = len(p["text"].replace(" ", ""))
        qualifying_body.append({
            "text": p["text"],
            "word_count": word_count,
            "char_count": char_count
        })

if qualifying_body:
    # (i) highest word count
    highest = max(qualifying_body, key=lambda x: x["word_count"])
    print("\nParagraph with highest word count:")
    print(highest["text"], "-", highest["word_count"], "words")

    # (ii) lowest word count (still above 5 since we already filtered)
    lowest = min(qualifying_body, key=lambda x: x["word_count"])
    print("\nParagraph with lowest word count (above 5):")
    print(lowest["text"], "-", lowest["word_count"], "words")

    # (iii) mean word count
    mean_wc = round(sum(p["word_count"] for p in qualifying_body) / len(qualifying_body), 2)
    print("\nMean word count across qualifying body paragraphs:", mean_wc)
else:
    print("\nNo body paragraphs with more than 5 words were found.")


# (d) Build a nested dictionary structured as { style_name: { "count": N, "texts": [ list ] } } for all styles found. Sort the outer dictionary by count in descending order. If a style has more than 2 paragraphs, show only the first 2 texts. Pretty-print the result.

style_dict = {}

for p in paragraphs_data:
    style = p["style"]
    if style not in style_dict:
        style_dict[style] = {"count": 0, "texts": []}

    style_dict[style]["count"] += 1
    style_dict[style]["texts"].append(p["text"])

# now trim texts down to first 2 if there's more than 2
for style in style_dict:
    if style_dict[style]["count"] > 2:
        style_dict[style]["texts"] = style_dict[style]["texts"][:2]

# sort by count descending -- using sorted() with a lambda on the count
sorted_styles = sorted(style_dict.items(), key=lambda x: x[1]["count"], reverse=True)
sorted_style_dict = OrderedDict(sorted_styles)

print("\nStyle breakdown (sorted by count):")
for style, info in sorted_style_dict.items():
    print(f"\n{style}: count = {info['count']}")
    for t in info["texts"]:
        print("   -", t)

# Task 5: DOCX Table Extraction and Pandas Integration

In [ ]:
# (a) Open a .docx file that contains at least one table. Iterate through doc.tables and extract each table as a list of lists: [[row1col1, row1col2, ...], [row2col1, ...]]. Print the total number of tables found and the dimensions (rows × columns) of each.

from docx import Document
import pandas as pd

docx_path = "Lab 02 Data Acquisition for NLP.docx"

doc = Document(docx_path)

all_tables = []

for table in doc.tables:
    table_data = []
    for row in table.rows:
        row_data = [cell.text for cell in row.cells]
        table_data.append(row_data)
    all_tables.append(table_data)

print("Total tables found:", len(all_tables))

for i, t in enumerate(all_tables, start=1):
    num_rows = len(t)
    num_cols = len(t[0]) if num_rows > 0 else 0  # just in case a table is somehow empty
    print(f"Table {i} dimensions: {num_rows} rows x {num_cols} columns")


# (b) Take the first table's data, treat the first row as the header and the remaining rows as data, and create a pandas DataFrame. Print the full DataFrame. Then count and print the number of empty or None cells per column.

first_table = all_tables[0]

header = first_table[0]
data_rows = first_table[1:]

df = pd.DataFrame(data_rows, columns=header)

print("\nFull DataFrame:")
print(df)

# treating both None and empty/whitespace strings as "empty"
empty_counts = df.apply(lambda col: col.apply(lambda x: x is None or str(x).strip() == "").sum())

print("\nEmpty or None cells per column:")
print(empty_counts)


# (c) Extract all paragraphs that appear before the first table in the document by iterating through doc.element.body children and stopping once a w:tbl element is encountered. Store these paragraphs in a list and print their count.

from docx.text.paragraph import Paragraph

pre_table_paragraphs = []

for child in doc.element.body:
    if child.tag.endswith('tbl'):   # this is a w:tbl element -> a table, so stop here
        break
    if child.tag.endswith('p'):     # this is a w:p element -> a paragraph
        para = Paragraph(child, doc)
        pre_table_paragraphs.append(para.text)

print("\nPre-table paragraph count:", len(pre_table_paragraphs))


# (d) Write both the pre-table paragraphs and the first table's data into a single file named docx_extracted.txt. The file must have clearly labeled sections: a --- PRE-TABLE PARAGRAPHS --- header followed by the paragraphs, then a --- TABLE DATA --- header followed by rows formatted as pipe-separated values, e.g., col1 | col2 | col3.

output_file = "docx_extracted.txt"

with open(output_file, "w") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for para_text in pre_table_paragraphs:
        f.write(para_text + "\n")

    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        line = " | ".join(row)  # pipe separated like asked
        f.write(line + "\n")

print(f"\n{output_file} written successfully.")

# quick re-read just to confirm it looks right
with open(output_file, "r") as f:
    print("\nFile contents:\n")
    print(f.read())

# Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

In [ ]:
# (a) Open two different .docx files (Document A and Document B). From each, extract all paragraphs whose style name starts with "Heading" into two separate lists. Using Python set operations, print: (i) headings common to both documents, (ii) headings only in Document A, and (iii) headings only in Document B.

from docx import Document
import re
from collections import Counter

docx_path_a = "documentA.docx"  # change to your file names
docx_path_b = "documentB.docx"

doc_a = Document(docx_path_a)
doc_b = Document(docx_path_b)

headings_a = [p.text for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

# using sets since we need common/only-in-A/only-in-B comparisons
set_a = set(headings_a)
set_b = set(headings_b)

common_headings = set_a & set_b
only_in_a = set_a - set_b
only_in_b = set_b - set_a

print("Headings common to both documents:", common_headings)
print("\nHeadings only in Document A:", only_in_a)
print("\nHeadings only in Document B:", only_in_b)


# (b) From each document, extract all "Normal" style paragraphs. Merge both lists. Remove exact duplicate paragraphs by comparing stripped, lowercased text. Print the total count before and after deduplication.

normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal"]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal"]

merged_normal = normal_a + normal_b

count_before_dedup = len(merged_normal)

# dedup by comparing stripped/lowercased text, but keep the original text
seen = set()
deduped_paragraphs = []

for para in merged_normal:
    key = para.strip().lower()
    if key not in seen and key != "":  # skip blanks too, they're not real duplicates to worry about but no point keeping them as "paragraphs"
        seen.add(key)
        deduped_paragraphs.append(para)

count_after_dedup = len(deduped_paragraphs)

print("\nBody paragraph count before deduplication:", count_before_dedup)
print("Body paragraph count after deduplication:", count_after_dedup)


# (c) Tokenize every paragraph in the deduplicated list into words (lowercase, punctuation removed via re.sub()). Build a combined word frequency dictionary using collections.Counter. Remove at least 15 stopwords. Display the top 20 most frequent words.

stopwords = {
    "the", "is", "in", "and", "to", "of", "a", "an", "it", "for",
    "on", "with", "as", "was", "were", "are", "this", "that", "be", "by"
}

all_tokens = []

for para in deduped_paragraphs:
    cleaned = re.sub(r'[^a-zA-Z\s]', '', para).lower()
    words = cleaned.split()
    all_tokens.extend(words)

filtered_tokens = [w for w in all_tokens if w not in stopwords]

word_freq = Counter(filtered_tokens)

top_20 = word_freq.most_common(20)

print("\nTop 20 most frequent words:")
for word, count in top_20:
    print(word, "-", count)


# (d) Compute and print a summary containing: (i) total headings across both documents, (ii) total unique body paragraphs after deduplication, (iii) total unique word types (vocabulary size), (iv) average paragraph length in words rounded to 2 decimal places, and (v) the single longest paragraph by word count from the combined set.

total_headings = len(headings_a) + len(headings_b)

total_unique_body_paragraphs = len(deduped_paragraphs)

# vocabulary size = number of unique word types, using the filtered tokens from (c)
vocab_size = len(set(filtered_tokens))

# average paragraph length based on word counts of the deduped paragraphs (using raw text, not filtered tokens)
para_word_counts = [len(p.split()) for p in deduped_paragraphs]
avg_para_length = round(sum(para_word_counts) / len(para_word_counts), 2) if para_word_counts else 0

# longest paragraph by word count
if deduped_paragraphs:
    longest_para = max(deduped_paragraphs, key=lambda p: len(p.split()))
else:
    longest_para = None

print("\n--- SUMMARY ---")
print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs after deduplication:", total_unique_body_paragraphs)
print("Total unique word types (vocabulary size):", vocab_size)
print("Average paragraph length in words:", avg_para_length)
print("Longest paragraph by word count:")
print(longest_para)

# Task 7: Multi-Endpoint API Data Joining and Analysis

In [ ]:
# (a) Send GET requests to both endpoints below. For each, verify the HTTP status code is exactly 200; if not, raise a ConnectionError with the actual status code in the message. Parse both responses as JSON lists.
# https://jsonplaceholder.typicode.com/users
# https://jsonplaceholder.typicode.com/posts

import requests

users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"

users_response = requests.get(users_url)
if users_response.status_code != 200:
    raise ConnectionError(f"Failed to fetch users, status code: {users_response.status_code}")

posts_response = requests.get(posts_url)
if posts_response.status_code != 200:
    raise ConnectionError(f"Failed to fetch posts, status code: {posts_response.status_code}")

users = users_response.json()
posts = posts_response.json()

print("Users fetched:", len(users))
print("Posts fetched:", len(posts))


# (b) Join the two datasets using id from users and userId from posts. Build a dictionary structured as: { user_name: { "email": "...", "post_count": N, "titles": [ list of post titles ] } }. Print the result for the first 3 users only.

joined_data = {}

for user in users:
    user_id = user["id"]
    user_name = user["name"]

    # grabbing all posts that belong to this user
    user_posts = [p for p in posts if p["userId"] == user_id]
    titles = [p["title"] for p in user_posts]

    joined_data[user_name] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": titles
    }

print("\nFirst 3 users:")
for name in list(joined_data.keys())[:3]:
    print(f"\n{name}:")
    print(joined_data[name])


# (c) From the joined dictionary, find the user who has written the most posts. Print their: (i) full name, (ii) email address, (iii) company name, and (iv) all post titles numbered.

# using max() on the dict items with post_count as the key
top_user_name = max(joined_data, key=lambda name: joined_data[name]["post_count"])
top_user_info = joined_data[top_user_name]

# need to go back to the original users list to grab the company name, since we didn't store it in joined_data
top_user_full = next(u for u in users if u["name"] == top_user_name)

print("\n--- USER WITH MOST POSTS ---")
print("Name:", top_user_name)
print("Email:", top_user_info["email"])
print("Company:", top_user_full["company"]["name"])

print("Post titles:")
for i, title in enumerate(top_user_info["titles"], start=1):
    print(f"{i}. {title}")


# (d) Filter users where either of the following is true: (i) company.bs field contains the word "synergies" (case-insensitive), or (ii) address.city has more than 8 characters. Print the name, city, and post count for all matching users.

print("\n--- MATCHING USERS ---")

for user in users:
    bs_text = user["company"]["bs"].lower()
    city = user["address"]["city"]

    condition_bs = "synergies" in bs_text
    condition_city = len(city) > 8

    if condition_bs or condition_city:
        post_count = joined_data[user["name"]]["post_count"]
        print(f"Name: {user['name']} | City: {city} | Post count: {post_count}")

# Task 8: To-Do Completion Rate Analysis Across Users

In [ ]:
# (a) Fetch data from the endpoint below and parse it into a Python list. Validate that: (i) the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at least one record. If any validation fails, print a meaningful error and exit. Print the total number of todo records retrieved.
# https://jsonplaceholder.typicode.com/todos

import requests
import sys

todos_url = "https://jsonplaceholder.typicode.com/todos"
users_url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(todos_url)

# (i) check status is 200
if response.status_code != 200:
    print(f"Error: request failed with status code {response.status_code}")
    sys.exit()

todos = response.json()

# (ii) check it's actually a list
if not isinstance(todos, list):
    print("Error: expected a list but got something else.")
    sys.exit()

# (iii) check it's not empty
if len(todos) == 0:
    print("Error: todo list is empty, nothing to work with.")
    sys.exit()

print("Total todo records retrieved:", len(todos))


# (b) Group todos by userId. For each user, compute and store: { userId: { "total": N, "completed": N, "incomplete": N } }. Print the result for all 10 users. For each user, verify that completed + incomplete == total and print True or False accordingly.

user_stats = {}

for todo in todos:
    uid = todo["userId"]

    if uid not in user_stats:
        user_stats[uid] = {"total": 0, "completed": 0, "incomplete": 0}

    user_stats[uid]["total"] += 1
    if todo["completed"]:
        user_stats[uid]["completed"] += 1
    else:
        user_stats[uid]["incomplete"] += 1

print("\nStats per user:")
for uid in sorted(user_stats.keys()):
    stats = user_stats[uid]
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats} -> completed + incomplete == total: {check}")


# (c) Compute the completion rate for each user as completed / total × 100 rounded to 1 decimal place. Print: (i) the userId with the highest completion rate and its value, (ii) the userId with the lowest completion rate and its value, and (iii) the overall average completion rate across all users.

completion_rates = {}

for uid, stats in user_stats.items():
    rate = round((stats["completed"] / stats["total"]) * 100, 1)
    completion_rates[uid] = rate

# (i) highest
highest_uid = max(completion_rates, key=lambda u: completion_rates[u])
print("\nHighest completion rate -> User", highest_uid, ":", completion_rates[highest_uid], "%")

# (ii) lowest
lowest_uid = min(completion_rates, key=lambda u: completion_rates[u])
print("Lowest completion rate -> User", lowest_uid, ":", completion_rates[lowest_uid], "%")

# (iii) overall average
avg_rate = round(sum(completion_rates.values()) / len(completion_rates), 1)
print("Overall average completion rate:", avg_rate, "%")


# (d) Fetch https://jsonplaceholder.typicode.com/users and build a lookup dictionary { userId: user_name }. Print a formatted table with columns: Rank | Name | Total Tasks | Completed | Completion Rate %, sorted by completion rate in descending order. Use f-strings or str.ljust() to align columns neatly.

users_response = requests.get(users_url)

if users_response.status_code != 200:
    print(f"Error: failed to fetch users, status code {users_response.status_code}")
    sys.exit()

users = users_response.json()

user_lookup = {u["id"]: u["name"] for u in users}

# sort userIds by completion rate, highest first
sorted_uids = sorted(completion_rates, key=lambda u: completion_rates[u], reverse=True)

print("\n--- COMPLETION RATE TABLE ---")
header = f"{'Rank':<6}{'Name':<25}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %':<18}"
print(header)
print("-" * len(header))

for rank, uid in enumerate(sorted_uids, start=1):
    name = user_lookup.get(uid, "Unknown")
    total = user_stats[uid]["total"]
    completed = user_stats[uid]["completed"]
    rate = completion_rates[uid]

    row = f"{rank:<6}{name:<25}{total:<14}{completed:<12}{rate:<18}"
    print(row)

# Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [ ]:
# (a) Fetch data from https://jsonplaceholder.typicode.com/users. Each user contains nested objects (address with geo, and company). Write a function flatten_user(user: dict) → dict that returns a single-level dictionary with exactly these keys: id, name, username, email, city, zipcode, lat, lng, company_name, company_catchphrase. Apply this function to all 10 users and print the resulting flattened list.

import requests
import pandas as pd
import math

users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"
comments_url = "https://jsonplaceholder.typicode.com/comments"

users = requests.get(users_url).json()

def flatten_user(user: dict) -> dict:
    # just pulling everything out of the nested address/geo/company dicts into one flat dict
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }

flattened_users = [flatten_user(u) for u in users]

print("Flattened user list:")
for u in flattened_users:
    print(u)


# (b) Create a pandas DataFrame from the flattened user list and display it in full. Then: (i) cast the lat and lng columns to float, (ii) add a new column distance_from_origin computed as sqrt(lat² + lng²) using the math module applied row-by-row, and (iii) print the user with the greatest distance from origin.

df = pd.DataFrame(flattened_users)

print("\nFull DataFrame:")
print(df)

# (i) lat/lng come back as strings from the API, so casting to float
df["lat"] = df["lat"].astype(float)
df["lng"] = df["lng"].astype(float)

# (ii) distance from origin, row by row using math.sqrt
df["distance_from_origin"] = df.apply(lambda row: math.sqrt(row["lat"]**2 + row["lng"]**2), axis=1)

print("\nDataFrame with distance_from_origin:")
print(df)

# (iii) user with greatest distance
farthest_user = df.loc[df["distance_from_origin"].idxmax()]
print("\nUser with greatest distance from origin:")
print(farthest_user)


# (c) Using the lat column, classify users into two groups: (i) Northern Hemisphere where lat > 0, and (ii) Southern Hemisphere where lat < 0. Print the names of users in each group. Then determine which hemisphere has the higher average distance_from_origin.

northern = df[df["lat"] > 0]
southern = df[df["lat"] < 0]

print("\nNorthern Hemisphere users:", list(northern["name"]))
print("Southern Hemisphere users:", list(southern["name"]))

# handling the case where a group might end up empty, just to be safe
north_avg = northern["distance_from_origin"].mean() if len(northern) > 0 else 0
south_avg = southern["distance_from_origin"].mean() if len(southern) > 0 else 0

print(f"\nNorthern Hemisphere avg distance: {north_avg:.2f}")
print(f"Southern Hemisphere avg distance: {south_avg:.2f}")

if north_avg > south_avg:
    print("Northern Hemisphere has the higher average distance.")
elif south_avg > north_avg:
    print("Southern Hemisphere has the higher average distance.")
else:
    print("Both hemispheres have the same average distance.")


# (d) Fetch both endpoints below. For each post, count its number of comments by matching postId in comments against id in posts. Build a sorted list of tuples: (post_id, post_title[:40], comment_count) in descending order by comment count. Print the top 5 most-commented and bottom 5 least-commented posts.
# https://jsonplaceholder.typicode.com/posts
# https://jsonplaceholder.typicode.com/comments

posts = requests.get(posts_url).json()
comments = requests.get(comments_url).json()

post_comment_data = []

for post in posts:
    post_id = post["id"]
    # counting how many comments have this postId
    comment_count = sum(1 for c in comments if c["postId"] == post_id)
    post_comment_data.append((post_id, post["title"][:40], comment_count))

# sort by comment count, descending
post_comment_data.sort(key=lambda x: x[2], reverse=True)

print("\nTop 5 most-commented posts:")
for entry in post_comment_data[:5]:
    print(entry)

print("\nBottom 5 least-commented posts:")
for entry in post_comment_data[-5:]:
    print(entry)